# Alert Retrieval Service

This notebook demonstrates how to retrieve Rubin alert packets from the RSP Alert retrieval service (Herald).

Alerts can be retrieved in three formats:

- **Avro OCF** (default): Self-describing binary with schema embedded
- **FITS**: multi-extension FITS file
- **JSON**: Deserialised alert record

All endpoints require authentication and accept the alert ID via the `ID` query parameter.

## Setup

In [ ]:
import base64
import io
import fastavro
import matplotlib.pyplot as plt
import numpy as np
from astropy.io import fits
from lsst.rsp import RSPClient, get_service_url

ALERT_ID = "170112073844916273"
url = get_service_url("alerts", "prompt")
client = RSPClient("")

## JSON format

The JSON response is the full deserialised alert record.

In [ ]:
r = await client.get(url, params={"ID": ALERT_ID, "RESPONSEFORMAT": "json"})
r.raise_for_status()

alert = r.json()
print(f"Top-level keys: {list(alert.keys())}")

In [ ]:
src = alert["diaSource"]
print(f"diaSourceId:     {src['diaSourceId']}")
print(f"RA / Dec:        {src['ra']:.6f}  {src['dec']:.6f}")
print(f"MJD (TAI):       {src['midpointMjdTai']:.6f}")
print(f"psfFlux (nJy):   {src['psfFlux']}")
print(f"Band:            {src['band']}")

# Cutout stamps are base64-encoded bytes in the JSON response
cutout_b64 = alert.get("cutoutDifference")
if cutout_b64:
    cutout_bytes = base64.b64decode(cutout_b64)
    print(f"\ncutoutDifference: {len(cutout_bytes)} bytes (base64-decoded)")

## Avro OCF format

The default response is an Avro OCF container file with the schema embedded in the file header.

In [ ]:
r = await client.get(url, params={"ID": ALERT_ID})
r.raise_for_status()

print(f"Content-Type: {r.headers['content-type']}")
print(f"Size: {len(r.content):,} bytes")

In [ ]:
# fastavro.reader reads the embedded schema automatically
records = list(fastavro.reader(io.BytesIO(r.content)))
record = records[0]

print(f"Schema name:  {fastavro.reader(io.BytesIO(r.content)).writer_schema['name']}")
print(f"Record count: {len(records)}")
print(f"diaSourceId:  {record['diaSource']['diaSourceId']}")
print(f"psfFlux:      {record['diaSource']['psfFlux']}")

## FITS format

The FITS response is a multi-extension file containing:

| Extension | Contents |
|---|---|
| `PRIMARY` | Empty; carries `TELESCOP` and `INSTRUME` headers |
| `ALERT` | Top-level scalar fields plus columns from `diaObject`, `ssObject`, or `mpc_orbits` (one row) |
| `DIFFIM` / `SCIENCE` / `TEMPLATE` | Cutout postage stamp images (if present) |
| `DIASOURCE` | Triggering source (row 0) and prior sources |
| `FORCEDPHOT` | Prior forced-photometry sources |
| `SSSOURCE` | Solar-system source (if present) |

In [ ]:
r = await client.get(url, params={"ID": ALERT_ID, "RESPONSEFORMAT": "fits"})
r.raise_for_status()

hdul = fits.open(io.BytesIO(r.content))
hdul.info()

In [ ]:
# Inspect the DIASOURCE table
diasource = hdul["DIASOURCE"]
print(f"DIASOURCE columns: {diasource.columns.names}")
print(f"Rows (triggering + prior sources): {len(diasource.data)}")
print()

for i, row in enumerate(diasource.data):
    label = "[trigger]" if row["trigger"] else "[prior]  "
    print(f"  {label} MJD={row['midpointMjdTai']:.5f}  psfFlux={row['psfFlux']:.2f} nJy  iau_id={row['iau_id'].strip()}")

### Cutout images from FITS response

The `DIFFIM`, `SCIENCE`, and `TEMPLATE` HDUs contain the postage stamp images embedded in the full alert FITS file.

In [ ]:
"""
# The following can be used to plot the potsage stamp images with matplotlib
import lsst.afw.display as afwDisplay
import lsst.afw.image as afwImage

afwDisplay.setDefaultBackend("matplotlib")

image_names = [
    hdu.name for hdu in hdul
    if hdu.data is not None and hdu.data.ndim == 2
]

if image_names:
    fig, axes = plt.subplots(1, len(image_names), figsize=(5 * len(image_names), 5))
    if len(image_names) == 1:
        axes = [axes]
    for ax, name in zip(axes, image_names):
        image = afwImage.ImageF(hdul[name].data.astype("float32"))
        plt.sca(ax)
        display = afwDisplay.Display(frame=fig)
        display.scale("asinh", "zscale")
        display.mtv(image, title=name)
    plt.suptitle(f"FITS images - alert {ALERT_ID}")
    plt.tight_layout()
    plt.show()
else:
    print("No image HDUs in this alert")
"""

## Cutout images

The `/cutouts` endpoint returns only the postage stamp images as a FITS file. 

In [ ]:
r = await client.get(f"{url}/cutouts", params={"ID": ALERT_ID})
r.raise_for_status()

cutouts = fits.open(io.BytesIO(r.content))
cutout_names = [hdu.name for hdu in cutouts if hdu.name != "PRIMARY"]
print(f"Available cutouts: {cutout_names}")

In [ ]:
"""
# Plot cutout images

import lsst.afw.display as afwDisplay
import lsst.afw.image as afwImage

afwDisplay.setDefaultBackend('matplotlib')

fig, axes = plt.subplots(1, len(cutout_names), figsize=(5 * len(cutout_names), 5))
if len(cutout_names) == 1:
  axes = [axes]                                                                                                                                                                                                

for ax, name in zip(axes, cutout_names):
  image = afwImage.ImageF(cutouts[name].data.astype("float32"))
  plt.sca(ax)
  display = afwDisplay.Display(frame=fig)
  display.scale('asinh', 'zscale')
  display.mtv(image, title=name)

t.suptitle(f"Cutouts - alert {ALERT_ID}")
plt.tight_layout()
plt.show()
"""

## Avro schema

The `/schema` endpoint returns the Avro schema used to encode a given alert.

In [ ]:
r = await client.get(f"{url}/schema", params={"ID": ALERT_ID})
r.raise_for_status()
schema = r.json()

print(f"Schema name:      {schema['name']}")
print(f"Schema namespace: {schema['namespace']}")
print(f"Top-level fields: {[f['name'] for f in schema['fields']]}")

## DataLink: discover related data products

The `/links` endpoint returns an IVOA DataLink VOTable listing all available
data products for the given alert.

In [ ]:
from astropy.io.votable import parse as parse_votable

r = await client.get(f"{url}/links", params={"ID": ALERT_ID})
r.raise_for_status()

vot = parse_votable(io.BytesIO(r.content))
table = vot.get_first_table().to_table()

for row in table:
    print(f"  {row['semantics']:<22} {row['content_type']:<30} {row['access_url']}")